# Data Quality Profiling
This notebook acts as a health check for our Data Engineering pipelines. We check for missing values, scraper success rates, and data anomalies.

In [ ]:
from google.cloud import bigquery
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

client = bigquery.Client()
project_id = os.environ.get('BIGQUERY_PROJECT_ID')
dataset_id = os.environ.get('BIGQUERY_DATASET')

In [ ]:
# Check for missing ratings across different platforms
query = f"""
SELECT 
    source,
    COUNT(*) as total_items,
    SUM(CASE WHEN avg_rating IS NULL THEN 1 ELSE 0 END) as missing_ratings,
    ROUND((SUM(CASE WHEN avg_rating IS NULL THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) as missing_rating_pct
FROM `{project_id}.{dataset_id}.int_clean_prices`
GROUP BY source
"""
df_dq = client.query(query).to_dataframe()
df_dq

In [ ]:
# Visualize Scraper Reliability
plt.figure(figsize=(8, 5))
sns.barplot(data=df_dq, x='source', y='missing_rating_pct', palette='mako')
plt.title('Percentage of Missing Ratings per Platform (Scraper Health)')
plt.xlabel('E-Commerce Platform')
plt.ylabel('% of Products Missing Ratings')
plt.axhline(20, color='red', linestyle='--', label='20% Alert Threshold')
plt.legend()
plt.show()